# 06. Multi-Model Serving

- metadata 기반 worker factory 이해
- LRU cache hit, miss, eviction 직접 확인

## 구현 코드

각 셀을 순서대로 실행해 component를 직접 정의한다.

In [ ]:
import json

from pydantic import BaseModel


class ModelMetadata(BaseModel):
  id: str
  name: str
  type: str
  framework: str
  version: str
  description: str


class ModelStore:
  def __init__(self, config_path: str):
    self.models: dict[str, ModelMetadata] = {}
    self._load_config(config_path)

  def _load_config(self, config_path: str):
    with open(config_path) as f:
      config = json.load(f)
    for model in config["models"]:
      self.models[model["id"]] = ModelMetadata(**model)

  def get_model(self, model_id: str) -> ModelMetadata | None:
    return self.models.get(model_id)

  def list_models(self) -> dict[str, ModelMetadata]:
    return self.models

In [ ]:
import os
import time
from abc import ABC, abstractmethod
from typing import Any

import numpy as np
import requests
import torch
import torchvision.transforms as transforms
from PIL import Image
from torchvision.models import MobileNet_V2_Weights, mobilenet_v2
from transformers import AutoModelForSequenceClassification, AutoTokenizer

TRITON_URL = os.getenv("TRITON_URL", "localhost:8009")


class ModelWorker(ABC):
  def __init__(self, model_metadata):
    self.model_metadata = model_metadata
    self.model: torch.nn.Module | None = None
    started = time.perf_counter()
    self._load_model()
    self.load_seconds = time.perf_counter() - started

  @abstractmethod
  def _load_model(self): ...

  @abstractmethod
  def predict(self, input_data: Any) -> dict[str, Any]: ...


class TransformerWorker(ModelWorker):
  def __init__(self, model_metadata):
    self.tokenizer: AutoTokenizer | None = None
    super().__init__(model_metadata)

  def _load_model(self):
    if self.model is None:
      self.model = AutoModelForSequenceClassification.from_pretrained(self.model_metadata.name)
      self.tokenizer = AutoTokenizer.from_pretrained(self.model_metadata.name)
      self.model.eval()

  def predict(self, input_data: Any) -> dict[str, Any]:
    inputs = self.tokenizer(input_data, return_tensors="pt", padding=True, truncation=True)
    with torch.no_grad():
      outputs = self.model(**inputs)
    predictions = torch.softmax(outputs.logits, dim=-1)
    return {"predictions": predictions.tolist()}


class TorchVisionWorker(ModelWorker):
  def __init__(self, model_metadata):
    self.transform: transforms.Compose | None = None
    super().__init__(model_metadata)

  def _load_model(self):
    if self.model is None:
      self.model = mobilenet_v2(weights=MobileNet_V2_Weights.DEFAULT)
      self.model.eval()
      self.transform = transforms.Compose(
        [
          transforms.Resize(256),
          transforms.CenterCrop(224),
          transforms.ToTensor(),
          transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ]
      )

  def predict(self, input_data: Any) -> dict[str, Any]:
    image = Image.open(input_data).convert("RGB") if isinstance(input_data, str) else input_data
    image_tensor = self.transform(image).unsqueeze(0)
    with torch.no_grad():
      outputs = self.model(image_tensor)
    predictions = torch.softmax(outputs, dim=1)
    top = torch.topk(predictions, k=5)
    return {"top5_index": top.indices.tolist(), "top5_prob": top.values.tolist()}


class TritonWorker(ModelWorker):
  def __init__(self, model_metadata):
    import tritonclient.http as httpclient

    self.httpclient = httpclient
    self.triton_url = TRITON_URL
    self.client = httpclient.InferenceServerClient(url=self.triton_url)
    super().__init__(model_metadata)

  def _load_model(self):
    load_url = f"http://{self.triton_url}/v2/repository/models/{self.model_metadata.name}/load"
    response = requests.post(load_url, timeout=120)
    if response.status_code != 200:
      raise RuntimeError(f"failed to load model on triton: {response.text}")
    if not self.client.is_model_ready(self.model_metadata.name):
      raise RuntimeError("model is not ready after loading")

  def predict(self, input_data: dict[str, Any]) -> dict[str, Any]:
    inputs = []
    for name, data in input_data.items():
      if isinstance(data, np.ndarray):
        array = data.astype(np.float32)
      else:
        array = np.array(data["data"], dtype=np.float32).reshape(data["shape"])
      tensor = self.httpclient.InferInput(name, array.shape, "FP32")
      tensor.set_data_from_numpy(array)
      inputs.append(tensor)

    output_name = "fc6_1"
    response = self.client.infer(
      model_name=self.model_metadata.name,
      inputs=inputs,
      outputs=[self.httpclient.InferRequestedOutput(output_name)],
    )
    scores = response.as_numpy(output_name).reshape(-1)
    top5 = np.argsort(scores)[-5:][::-1]
    return {"top5_index": top5.tolist(), "top5_score": scores[top5].tolist()}

  def unload(self):
    unload_url = f"http://{self.triton_url}/v2/repository/models/{self.model_metadata.name}/unload"
    try:
      requests.post(unload_url, timeout=60)
    except Exception:
      pass

  def __del__(self):
    self.unload()

In [ ]:
WORKER_TYPES = {
  "transformers": TransformerWorker,
  "torchvision": TorchVisionWorker,
  "triton": TritonWorker,
}


class ModelEngine:
  def __init__(self):
    self.workers: dict[str, ModelWorker] = {}

  def get_worker(self, model_id: str) -> ModelWorker | None:
    return self.workers.get(model_id)

  def create_worker(self, model_metadata: ModelMetadata) -> ModelWorker:
    if model_metadata.id not in self.workers:
      worker_type = WORKER_TYPES.get(model_metadata.framework)
      if worker_type is None:
        raise ValueError(f"unsupported framework: {model_metadata.framework}")
      self.workers[model_metadata.id] = worker_type(model_metadata)
    return self.workers[model_metadata.id]

  def delete_worker(self, model_id: str):
    worker = self.workers.pop(model_id, None)
    if worker is not None and hasattr(worker, "unload"):
      worker.unload()

In [ ]:
import threading
import time
from collections import OrderedDict
from typing import Any


class ModelManager:
  def __init__(self, model_store: ModelStore, max_models: int = 2):
    self.model_store = model_store
    self.max_models = max_models
    self.model_cache: OrderedDict[str, ModelWorker] = OrderedDict()
    self.model_engine = ModelEngine()
    self.lock = threading.Lock()
    self.metrics = {"hits": 0, "misses": 0, "evictions": 0, "load_seconds": 0.0}

  def get_model_worker(self, model_id: str) -> ModelWorker | None:
    with self.lock:
      if model_id in self.model_cache:
        self.model_cache.move_to_end(model_id)
        self.metrics["hits"] += 1
        return self.model_engine.get_worker(model_id)

      model_metadata = self.model_store.get_model(model_id)
      if not model_metadata:
        return None

      self.metrics["misses"] += 1
      if len(self.model_cache) >= self.max_models:
        evicted_id, _ = self.model_cache.popitem(last=False)
        self.model_engine.delete_worker(evicted_id)
        self.metrics["evictions"] += 1

      started = time.perf_counter()
      worker = self.model_engine.create_worker(model_metadata)
      self.metrics["load_seconds"] += time.perf_counter() - started
      self.model_cache[model_id] = worker
      return worker

  def list_loaded_models(self) -> dict[str, str]:
    with self.lock:
      return {mid: w.model_metadata.name for mid, w in self.model_cache.items()}

  def stats(self) -> dict[str, Any]:
    with self.lock:
      total = self.metrics["hits"] + self.metrics["misses"]
      return {
        "max_models": self.max_models,
        "cached": list(self.model_cache.keys()),
        "hit_rate": round(self.metrics["hits"] / total, 3) if total else None,
        **self.metrics,
      }

## model download 없는 LRU 실험

가짜 engine을 연결해 cache 정책만 분리해서 실행한다.

In [ ]:
class FakeWorker:
  def __init__(self, metadata):
    self.model_metadata = metadata


class FakeEngine:
  def __init__(self):
    self.workers = {}

  def get_worker(self, model_id):
    return self.workers.get(model_id)

  def create_worker(self, metadata):
    worker = FakeWorker(metadata)
    self.workers[metadata.id] = worker
    return worker

  def delete_worker(self, model_id):
    self.workers.pop(model_id, None)


from pathlib import Path

config_path = Path("config/models.json")
if not config_path.exists():
  config_path = Path("06_multimodel/config/models.json")
store = ModelStore(str(config_path))
manager = ModelManager(store, max_models=2)
manager.model_engine = FakeEngine()
model_ids = list(store.models)[:3]
for model_id in [*model_ids, *model_ids]:
  manager.get_model_worker(model_id)
manager.stats()

## 퀴즈

코드를 다시 보지 않고 먼저 답해본다.

1. cache 크기 2에 model 3개 요청이 round-robin으로 들어오면 왜 thrashing이 발생하는가?
2. TritonWorker가 Triton에 위임하는 기능은 무엇인가?

<details>
<summary>정답과 해설 보기</summary>

1. 다음 차례에 필요한 model이 항상 직전 eviction 대상이 된다. 거의 모든 요청이 cache miss와 model reload를 일으킨다.
2. model repository의 load·unload, framework별 추론 실행, tensor 기반 inference protocol과 accelerator 메모리 관리를 위임한다.

</details>